# Groq & OpenWeather experiment

This notebook serves the purpouse to get more familiar with Groq and OpenWeather and explore possible solutions that I will later dedicate myself to implement.

In [2]:
import os
import requests
from groq import Groq
from dotenv import load_dotenv
from datetime import datetime
import json

In [3]:
loaded = load_dotenv()
print("Env file loaded:", loaded)

Env file loaded: True


In [4]:
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')
Groq.api_key = os.getenv("GROQ_API_KEY")
print("Groq key found:", bool(Groq.api_key))
print(f"OpenWeather API: {'Loaded' if OPENWEATHER_API_KEY else 'Missing'}")
print(f"Groq API: {'Loaded' if Groq.api_key else 'Missing'}")

Groq key found: True
OpenWeather API: Loaded
Groq API: Loaded


In [5]:
if not Groq.api_key:
    raise ValueError("Groq API key not found!")

Initial thought is that prompts by users are ambiguous, and models don't clearly understand what for example tomorrow means. They are normally not trained to tell time. So my initial thinking is to parse information about user's location and time to add context to final prompt.

In [6]:
def get_current_datetime():
    return datetime.now()

get_current_datetime()

datetime.datetime(2025, 10, 6, 4, 30, 2, 101083)

In [7]:
IPINFO_API_KEY = os.getenv('IPINFO_API_KEY')
print(f"IPInfo API: {'Loaded' if IPINFO_API_KEY else 'Missing'}")

IPInfo API: Loaded


## Requesting system and location information

In [8]:
IPINFO_url = f"https://ipinfo.io/json?token={IPINFO_API_KEY}"
location_info = requests.get(IPINFO_url)
data = location_info.json()
print(data)

{'ip': '188.120.119.237', 'hostname': '188-120-119-237.dynamic.a1.rs', 'city': 'Novi Beograd', 'region': 'Central Serbia', 'country': 'RS', 'loc': '44.8056,20.4242', 'org': 'AS44143 A1 Srbija d.o.o', 'postal': '11031', 'timezone': 'Europe/Belgrade'}


This request didn't gave me precise information about my location. Currently I am in Smederevo, Serbia. Pančevo is not far from Smederevo, so the weather is not much different. Since I am building an MVP, I will stick with this option for now.

In [9]:
def get_user_location():
    try:
        response = requests.get(IPINFO_url)
        data = response.json()
        city = data.get("city")
        country = data.get("country")
        loc = data.get("loc")  # latitude and longitude
        lat, lon = map(float, loc.split(","))
        timezone = data.get("timezone")
        return {"city": city, "country": country, "lat": lat, "lon": lon, "timezone": timezone}
    except Exception as e:
        print("Could not get location:", e)
        return None
    
get_user_location()

{'city': 'Novi Beograd',
 'country': 'RS',
 'lat': 44.8056,
 'lon': 20.4242,
 'timezone': 'Europe/Belgrade'}

In [10]:
def contextual_data():
    dt = get_current_datetime()
    loc = get_user_location() or {}
    data = {
        "local_datetime": {
            "year": dt.year,
            "month": dt.month,
            "day": dt.day,
            "hour": dt.hour,
            "minute": dt.minute,
            "second": dt.second,
            "iso": dt.isoformat()
        },
        "location": {
            "city": loc.get("city", None),
            "country": loc.get("country", None),
            "latitude": loc.get("lat"),
            "longitude": loc.get("lon")
        }
    }

    return data
c = contextual_data()
c

{'local_datetime': {'year': 2025,
  'month': 10,
  'day': 6,
  'hour': 4,
  'minute': 30,
  'second': 26,
  'iso': '2025-10-06T04:30:26.915474'},
 'location': {'city': 'Novi Beograd',
  'country': 'RS',
  'latitude': 44.8056,
  'longitude': 20.4242}}

## Familiarizing with Groq

The next step I imagined is to connect context data with user's prompt ex: user might ask about weather info in Madrid while being located in Berlin. The model needs to know what data should prioritize. When it comes to other data on cloathing, good medical practices, activity, dining suggestions etc.

In [11]:
from openai import OpenAI
# this part of code is taken from Groq documentation https://console.groq.com/docs/overview
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

prompt = "Was today a good day to go for a walk?"

response = client.responses.create(
    input=prompt,
    model="openai/gpt-oss-20b",
)
print(response.output_text)



I’m not able to pull in today’s live weather data, so I can’t say for sure whether it’s a good walk‑day for you.  
Here are a few quick checks you can do on your phone or a local weather website/app:

| Factor | What to look for | Why it matters |
|--------|------------------|----------------|
| **Temperature** | Comfortable (usually 50–70°F / 10–21°C for most people) | Too hot or too cold can make walking uncomfortable or risky. |
| **Precipitation** | Dry or light rain (and you have appropriate gear) | Heavy rain, snow, or sleet can be hazardous and less enjoyable. |
| **Wind** | Light or moderate | Strong winds can chill you and make walking harder. |
| **Air quality** | Good or moderate | Poor air can irritate lungs, especially if you’re walking for exercise. |
| **Humidity** | Low–moderate | High humidity can make you feel hotter and sweaty. |

If you check your local forecast and everything looks favorable, it’s probably a nice day to go for a walk. If there’s a storm, high wind,

This was a test prompt. I expected it to give any response instead of being this honest. But this response is very useful to take into consideration what can I take care of when developing solution.  I will copy response here:
| Check | What to look for | Why it matters | Quick way to find it |
|-------|-----------------|----------------|--------------------|
| **Weather** | Temperature, wind speed, precipitation | Comfortable temperatures (roughly 50–75 °F / 10–24 °C) are ideal for most people. Wind‑tough or rain‑heavy days can be less enjoyable (or even unsafe). | Weather app (iOS/Android), AccuWeather, Weather.com, or local news. |
| **Air quality** | AQI (Air Quality Index) and main pollutants (PM2.5, PM10, O₃) | Good air (AQI < 50) means you’ll breathe easily; moderate (50‑100) is okay for most, but people with asthma or heart conditions may want to stay inside. | AirNow.gov, AQICN.org, or a local air‑quality widget. |
| **Sunrise / Sunset** | Time of day | If you’re planning a longer walk or want daylight, you’ll want to finish before dark. | Sunrise Sunset (app or website) or simply check your phone’s calendar. |
| **Crowd levels** | Are popular parks or trails busy? | Less crowded means more space, quieter, and easier to keep social distance. | Google Maps “Popular times,” park websites, or local community groups on Facebook/Nextdoor. |
| **Trail/park conditions** | Closed paths, maintenance, or wildlife alerts | Closed or hazardous paths may cancel a walk. | Park or trail website, or call the park office. |
| **Noise / Pollution** | Traffic or construction noise | Can affect enjoyment; some prefer quieter areas. | Local news or city council announcements. |
| **Personal health** | Any recent illness, injury, or medication | If you’re not feeling well, it might be better to stay in. | Listen to your body; consult your doctor if uncertain. |


In [ ]:
def ask_groq(prompt):
    response = client.responses.create(
        input=prompt,
        model="openai/gpt-oss-20b",
    )
    return response.output_text


This markdown are suggestions from Groq
| Category | What it Covers | Why It Helps Users | Typical Data Sources / Features |
|----------|----------------|--------------------|---------------------------------|
| **Weather Snapshot** | Current temps, humidity, wind, UV index, chance of precipitation, visibility, sunrise/sunset, day‑length. | Gives the *big picture* that drives every other suggestion. | NOAA, MET, OpenWeatherMap APIs; real‑time alerts. |
| **Activities & Outdoor Plans** | Hiking, biking, beach trips, park picnics, sports, garden work, indoor alternatives. | Matches user interests with the best‑timed activity. | Activity catalogs, geofenced POIs, user‑rated “best‑time” tags. |
| **Clothing & Gear** | Outfit layers, shoes, accessories, gear (umbrella, sunglasses, sunscreen, gloves). | Saves the hassle of “what to wear today?” | User profile (style, climate tolerance), weather‑linked wardrobe library. |
| **Transportation & Commute** | Traffic forecasts, public transit delays, bike‑share availability, weather‑safe routes (driving, walking, cycling). | Keeps the day running smoothly, even in bad weather. | Map APIs, transit feeds, real-time traffic. |
| **Food & Beverage** | Restaurant suggestions (indoor/outdoor), coffee shop recommendations, drinks (hot/cold) based on weather, special menu items. | Helps plan meals that fit the mood of the day. | Yelp/Foursquare API, weather-driven menu filters. |
| **Entertainment & Events** | Concerts, movies, festivals, indoor exhibitions, live streams. | Keeps users engaged whether the day is sunny or rainy. | Ticketing APIs, event calendars, user-interest tags. |
| **Health & Wellness** | UV-safety tips, hydration reminders, allergy alerts, exercise intensity. | Protects health while encouraging activity. | EPA allergens data, NASA UV index, health APIs. |
| **Finance & Deals** | Weather-driven sales, coupons for indoor/outdoor gear, energy-saving tips. | Adds value by turning weather into cost-saving opportunities. | Retail API, coupon aggregator. |
| **Home & Lifestyle** | Indoor projects (gardening, DIY), laundry schedules, cleaning tasks, smart-home device control. | Uses weather to decide what can be done inside or outside. | IoT integrations, home-automation APIs. |
| **Pet & Animal Care** | Pet walks, vet visits, pet-friendly spots, shelter advice. | Helps pet owners plan around weather constraints. | Pet-service APIs, veterinary clinics. |
| **Travel & Commute Planning** | Flight status, train schedules, airport weather, layover suggestions. | Reduces travel stress linked to weather delays. | Flight/rail APIs, weather alerts for airports. |
| **Zodiac & Personalization** | Daily horoscope, numerology, “planet-aligned” suggestions for activities/clothing. | Adds a fun, optional “spiritual” layer for believers. | Astrology API, user preference toggles. |
| **Safety & Alerts** | Severe-weather warnings, evacuation routes, emergency contacts, health advisories. | Keeps users safe and informed. | NOAA severe-weather feeds, local emergency services. |
| **Eco & Green Living** | Carbon-footprint calculator for commute, sustainable shopping tips, plant-care reminders. | Encourages environmentally conscious choices. | Carbon-API, plant-care data. |

Basically Groq understands contextual data like they are written. I don't have to fruther explain it.

There are many things that can be implemented. I was initially considering including transportation, but coverage is limited in some APIs I explored. It could be a nice addition, but for now I will keep it simple. Transportation could be included in the context of advice, such as: choose a bike or walk because it’s nearby and the weather is nice; take the bus or drive because the air is polluted or rain is expected; or stay at home because of bad weather.

So, I will focus on general recommendations, such as activities in the user’s city or relevant suggestions for visitors. In future iterations, this app could become more specialized in certain directions. Currently, the output should include something like:

**1. Weather:** summary with alerts if extreme conditions are expected

**2. Clothing suggestions**

**3. Transportation suggestions:** type of transport if the user goes out

**4. Health advice:** supplementation, skin protection, hydration if the user goes out

**5. Activity suggestions:** depending on the user's availability — dine in, dine out, delivery, or prepare something at home

**6. Alerts:** e.g., if it’s a non-working day

**7. Other:** anything else relevant the model finds noteworthy

## Exploring OpenWeather

In the following lines I will explore the ways how to take needed data from OpenWeather.
First problem that I encounter is what if a user wants to have data about numerous locations and datetime instead of one?
So there should be function that parses informations and stores it.

In [15]:
def get_weather_now(lat, lon, api_key):
    """Minimal weather fetch - paste your API key and coordinates"""
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}"
    response = requests.get(url)
    data = response.json()
     # Print key info
    print(f"Location: {data['name']}")
    print(f"Temperature: {data['main']['temp']}°C")
    print(f"Feels like: {data['main']['feels_like']}°C")
    print(f"Weather: {data['weather'][0]['description']}")
    print(f"Humidity: {data['main']['humidity']}%")
    print(f"Wind: {data['wind']['speed']} m/s")
   
    return data

get_weather_now(44.8718, 20.6417, OPENWEATHER_API_KEY)

Location: Pančevo
Temperature: 282.52°C
Feels like: 280.76°C
Weather: moderate rain
Humidity: 91%
Wind: 3.26 m/s


{'coord': {'lon': 20.6417, 'lat': 44.8718},
 'weather': [{'id': 501,
   'main': 'Rain',
   'description': 'moderate rain',
   'icon': '10n'}],
 'base': 'stations',
 'main': {'temp': 282.52,
  'feels_like': 280.76,
  'temp_min': 282.52,
  'temp_max': 283.26,
  'pressure': 1015,
  'humidity': 91,
  'sea_level': 1015,
  'grnd_level': 1004},
 'visibility': 10000,
 'wind': {'speed': 3.26, 'deg': 303, 'gust': 7.62},
 'rain': {'1h': 3.99},
 'clouds': {'all': 100},
 'dt': 1759718002,
 'sys': {'type': 2,
  'id': 2037836,
  'country': 'RS',
  'sunrise': 1759725680,
  'sunset': 1759766970},
 'timezone': 7200,
 'id': 787237,
 'name': 'Pančevo',
 'cod': 200}

In [16]:
# this function is useful when data is parsed from user input
def city_to_coords(city, country, api_key):
    url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country}&limit=1&appid={api_key}"
    response = requests.get(url)
    data = response.json()
    if data:
        return data[0]['lat'], data[0]['lon']
    else:
        return None, None

city_to_coords("Madrid", "ES", OPENWEATHER_API_KEY)

(40.4167047, -3.7035825)

In [17]:
def get_weather_past(lat, lon, dt, api_key):
    """Fetch historical weather data for given coordinates and datetime (UNIX timestamp)"""
    url = f"https://api.openweathermap.org/data/2.5/onecall/timemachine?lat={lat}&lon={lon}&dt={dt}&appid={api_key}"
    response = requests.get(url)
    data = response.json()
    return data

get_weather_past(44.8718, 20.6417, 1696060800, OPENWEATHER_API_KEY)  # Example timestamp

{'cod': 401,
 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}

In [18]:
def get_weather_4_days(lat, lon, api_key):
    """Fetch 4-day weather forecast for given coordinates"""
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}"
    response = requests.get(url)
    data = response.json()
    return data

get_weather_4_days(44.8718, 20.6417, OPENWEATHER_API_KEY)

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1759719600,
   'main': {'temp': 282.52,
    'feels_like': 280.76,
    'temp_min': 282.52,
    'temp_max': 282.88,
    'pressure': 1015,
    'sea_level': 1015,
    'grnd_level': 1004,
    'humidity': 91,
    'temp_kf': -0.36},
   'weather': [{'id': 500,
     'main': 'Rain',
     'description': 'light rain',
     'icon': '10n'}],
   'clouds': {'all': 100},
   'wind': {'speed': 3.26, 'deg': 303, 'gust': 7.62},
   'visibility': 10000,
   'pop': 1,
   'rain': {'3h': 1.69},
   'sys': {'pod': 'n'},
   'dt_txt': '2025-10-06 03:00:00'},
  {'dt': 1759730400,
   'main': {'temp': 282.99,
    'feels_like': 280.88,
    'temp_min': 282.99,
    'temp_max': 283.93,
    'pressure': 1016,
    'sea_level': 1016,
    'grnd_level': 1005,
    'humidity': 90,
    'temp_kf': -0.94},
   'weather': [{'id': 500,
     'main': 'Rain',
     'description': 'light rain',
     'icon': '10d'}],
   'clouds': {'all': 100},
   'wind': {'speed': 4.16, 'deg': 305, 'g

In [19]:
def get_weather_16_days(lat, lon, api_key):
    """Fetch 16-day weather forecast for given coordinates"""
    url = f"https://api.openweathermap.org/data/2.5/forecast/daily?lat={lat}&lon={lon}&cnt=16&appid={api_key}"
    response = requests.get(url)
    data = response.json()

    return data

get_weather_16_days(44.8718, 20.6417, OPENWEATHER_API_KEY)

{'city': {'id': 787237,
  'name': 'Pančevo',
  'coord': {'lon': 20.6417, 'lat': 44.8718},
  'country': 'RS',
  'population': 76654,
  'timezone': 7200},
 'cod': '200',
 'message': 1.3102914,
 'cnt': 16,
 'list': [{'dt': 1759744800,
   'sunrise': 1759725680,
   'sunset': 1759766970,
   'temp': {'day': 286.38,
    'min': 281.8,
    'max': 289.02,
    'night': 281.8,
    'eve': 284.1,
    'morn': 282.72},
   'feels_like': {'day': 285.66,
    'night': 279.51,
    'eve': 283.39,
    'morn': 280.79},
   'pressure': 1019,
   'humidity': 73,
   'weather': [{'id': 502,
     'main': 'Rain',
     'description': 'heavy intensity rain',
     'icon': '10d'}],
   'speed': 6.83,
   'deg': 317,
   'gust': 10.45,
   'clouds': 97,
   'pop': 1,
   'rain': 14.1},
  {'dt': 1759831200,
   'sunrise': 1759812155,
   'sunset': 1759853260,
   'temp': {'day': 287.12,
    'min': 282.12,
    'max': 289.6,
    'night': 285.37,
    'eve': 288.04,
    'morn': 282.42},
   'feels_like': {'day': 286.14,
    'night': 284.

## Data prioritization

Here based on the user's prompt and contextual data, Groq tries to figure out what user wants and to produce needed information to proceed requesting data from OpenWeather.

In [ ]:
context_data = contextual_data()

## Final Request
Sending final prompt to model

## Chat-like implementation
In the following lines implementing chat like enviroment is explored.

## Discussing metrics

I am looking at ways how to evaluate my implementation